# Dataset Exploration

## Goal
Explore English/French parallel text before modeling.

## Setup
Select **Python (machine-translation-lab)** or the project `.venv` kernel. All code cells are intentionally TODO-only. Work through the [Hugging Face translation tutorial](https://huggingface.co/docs/transformers/tasks/translation). No training, data loading, or model loading happens automatically.

In [9]:
# TODO: Add your imports and confirm the selected Python interpreter.

from pathlib import Path
from datasets import load_dataset

## Steps

### 1. Loading OPUS Books
Use `Helsinki-NLP/opus_books`, configuration `en-fr`. Start with metadata or streaming if you want to avoid downloading the full split. Streaming datasets need different splitting methods than an in-memory Dataset.

In [ ]:
# TODO: Load the dataset when you are ready. Inspect available splits and features.

books = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr",
    cache_dir=str(Path.home() / ".cache" / "huggingface" / "datasets"),
)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})


In [11]:
books["train"][0]
{'id': '90560',
 'translation': {'en': 'But this lofty plateau measured only a few fathoms, and soon we reentered Our Element.',
  'fr': 'Mais ce plateau élevé ne mesurait que quelques toises, et bientôt nous fûmes rentrés dans notre élément.'}}

{'id': '90560',
 'translation': {'en': 'But this lofty plateau measured only a few fathoms, and soon we reentered Our Element.',
  'fr': 'Mais ce plateau élevé ne mesurait que quelques toises, et bientôt nous fûmes rentrés dans notre élément.'}}

### 2. Inspecting English/French pairs
The `translation` field holds the language pair. Inspect a small sample before processing more rows.

In [3]:
# TODO: Display a few pairs and describe their fields, languages, and alignment.  


print(books)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})


### 3. Training/test splits
The tutorial creates a holdout from the train split. Set a random seed. Consider whether nearby sentences or duplicate pairs could leak between splits.

In [4]:
# TODO: Create a reproducible split and check its sizes and overlap.


books = books["train"].train_test_split(test_size=0.2, seed=42)

print(books)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 101668
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 25417
    })
})


### 4. Dataset quality questions
Look for empty text, repeated pairs, length outliers, language mismatch, and imperfect alignment. Book prose may differ from retail text.

In [5]:
# TODO: Write checks and record the limitations you observe. 

books["train"][0]
{'id': '90560',
 'translation': {'en': 'But this lofty plateau measured only a few fathoms, and soon we reentered Our Element.',
  'fr': 'Mais ce plateau élevé ne mesurait que quelques toises, et bientôt nous fûmes rentrés dans notre élément.'}}

{'id': '90560',
 'translation': {'en': 'But this lofty plateau measured only a few fathoms, and soon we reentered Our Element.',
  'fr': 'Mais ce plateau élevé ne mesurait que quelques toises, et bientôt nous fûmes rentrés dans notre élément.'}}

## Checks
Restart the kernel and run your completed cells in order. Record what you checked, any assumptions, and remaining questions.

In [6]:
# TODO: Add your checks and observations. 

# 1. Check the split sizes
n_train = len(books["train"])
n_test = len(books["test"])

print("Training examples:", n_train)
print("Test examples:", n_test)
print(f"Test percentage: {n_test / (n_train + n_test):.1%}")

# 2. Check for empty text and repeated translation pairs
pair_sets = {}

for split in ["train", "test"]:
    pairs = [
        (row["translation"]["en"].strip(), row["translation"]["fr"].strip())
        for row in books[split]
    ]

    empty_count = sum(not en or not fr for en, fr in pairs)
    duplicate_count = len(pairs) - len(set(pairs))

    print(f"{split}: {empty_count} empty pairs, "
          f"{duplicate_count} repeated pairs")
    pair_sets[split] = set(pairs)

# 3. Check whether identical pairs appear in both splits
overlap = pair_sets["train"] & pair_sets["test"]
print("Pairs appearing in both train and test:", len(overlap))

Training examples: 101668
Test examples: 25417
Test percentage: 20.0%
train: 0 empty pairs, 461 repeated pairs
test: 0 empty pairs, 64 repeated pairs
Pairs appearing in both train and test: 119


## Next Steps
Save your observations and continue to the next notebook. After notebook 4, review the complete pipeline. Fine-tuning is optional later and is outside this setup.